# PrepareNeo4jBulkImport
This notebook uses metadata to prepare CSV Files for Neo4j Bulk Data Import

Author: Peter W. Rose (pwrose@ucsd.edu)

In [ ]:
import os
from os import walk
from os.path import join
import pandas as pd
import numpy as np
from utils import create_node_headers, create_relationship_headers, get_node_data_headers, get_relationship_data_headers, create_meta_node, create_meta_relationship

In [ ]:
# reload modules before executing user code
%load_ext autoreload
%autoreload 2

In [ ]:
# configure pandas dataframe
pd.options.display.max_rows = None  # display all rows
pd.options.display.max_columns = None  # display all columsns
pd.set_option("display.max_colwidth", None)

## Specify working directories

If NEO4J_HOME directory is not set, this notebook creates the example_results/import directory as a proxy for a Neo4j import directory.

In [ ]:
NEO4J_HOME = os.getenv("NEO4J_HOME", default="../example_results")

Location of Neo4j import directory

In [ ]:
NEO4J_IMPORT = os.path.join(NEO4J_HOME, "import")

Create the example_results/import directory for testing the example provided in this repo.

In [ ]:
if NEO4J_HOME == "../example_results":
    os.makedirs(os.path.join(NEO4J_IMPORT), exist_ok=True)

Locations of metadata files (Defaults are files for testing this notebook)

In [ ]:
METADATA = os.getenv("NEO4J_METADATA", default="../example_metadata/")
NODE_METADATA = os.path.join(METADATA, "nodes")
RELATIONSHIP_METADATA = os.path.join(METADATA, "relationships")

Locations of data files (Defaults are files for testing this notebook)

In [ ]:
DATA = os.getenv("NEO4J_DATA", default="../example_data/")
NODE_DATA = os.path.join(DATA, "nodes")
RELATIONSHIP_DATA = os.path.join(DATA, "relationships")

## Create headers from metadata files

Create the node name, the expected node file header, and the Neo4j header for bulk import from the metadata files.

In [ ]:
dirpath, _, filenames = next(walk(NODE_METADATA))
node_headers = [create_node_headers(dirpath, filename) for filename in filenames]

In [ ]:
node_metadata = pd.DataFrame(node_headers)

In [ ]:
node_metadata

Create the relationship name, the expected relatinonship file header, and the Neo4j header for bulk import from the metadata files.

In [ ]:
dirpath, _, filenames = next(walk(RELATIONSHIP_METADATA))
relationship_headers = [create_relationship_headers(dirpath, filename) for filename in filenames]

In [ ]:
relationship_metadata = pd.DataFrame(relationship_headers)

In [ ]:
relationship_metadata

## Add constraints and indices for Nodes

In [ ]:
def get_string_properties(row):
    node = row["node"]
    header = row["importHeader"]
    fields = header.split(",")
    fields = filter(lambda field: field.endswith(":string"), fields)
    string_properties = []
    
    for field in fields:
        field_name = field.split(":")[0]
        string_properties.append(field_name)

    return string_properties

In [ ]:
node_metadata["stringProperties"] = node_metadata.apply(get_string_properties, axis=1)

In [ ]:
def add_index(row):
    node = row["node"]
    properties = row["stringProperties"]
    indices = f"CREATE CONSTRAINT {node} FOR (n:{node}) REQUIRE n.identifier IS UNIQUE;"
    
    for prop in properties:
        indices += f"CREATE INDEX {node}_{prop} FOR (n:{node}) ON (n.{prop});"
        #indices += f"CREATE FULLTEXT INDEX FOR (n:{node}) ON EACH [n.{prop}];"

    return indices

In [ ]:
node_metadata["index"] = node_metadata.apply(add_index, axis=1)

In [ ]:
node_metadata.head()

## Get headers from data files

Node data files

In [ ]:
dirpath, _, filenames = next(walk(NODE_DATA))
csv_files = filter(lambda name: name.endswith(".csv"), filenames)
data_headers = [get_node_data_headers(dirpath, filename) for filename in csv_files]

In [ ]:
node_data = pd.DataFrame(data_headers)

In [ ]:
node_data

Relationship data files

In [ ]:
dirpath, _, filenames = next(walk(RELATIONSHIP_DATA))
csv_files = filter(lambda name: name.endswith(".csv"), filenames)
data_headers = [get_relationship_data_headers(dirpath, filename) for filename in csv_files]

In [ ]:
relationship_data = pd.DataFrame(data_headers)

In [ ]:
relationship_data

## Merge metadata with data

In [ ]:
matched_nodes = node_data.merge(node_metadata, on="node", how="outer")
matched_nodes["match"] = matched_nodes["dataHeader"] == matched_nodes["metadataHeader"]
matched_nodes.fillna("", inplace=True)

In [ ]:
matched_nodes

In [ ]:
mismatched_nodes = matched_nodes[(matched_nodes["match"] == False) & (matched_nodes["dataPath"] != "")]
mismatched_nodes

In [ ]:
if mismatched_nodes.shape[0] > 0:
    print("The following node data files do not match the metadata specification:")
    mismatched_nodes.to_csv(os.path.join(NEO4J_IMPORT, "mismatches_n.csv"), index=False)
    
mismatched_nodes

In [ ]:
matched_relationships = relationship_data.merge(relationship_metadata, on="relationship", how="outer")
matched_relationships["match"] = matched_relationships["dataHeader"] == matched_relationships["metadataHeader"]
matched_relationships.fillna("", inplace=True)
matched_relationships["fullRelationship"] = matched_relationships["source"] + "-" + matched_relationships["relationship"] + "-" + matched_relationships["target"]
matched_relationships

In [ ]:
mismatched_relationships = matched_relationships[(matched_relationships["match"] == False) & (matched_relationships["dataPath"] != "")]
if mismatched_relationships.shape[0] > 0:
    print("The following relationship data files do not match the metadata specification:")
    mismatched_relationships.to_csv(os.path.join(NEO4J_IMPORT, "mismatches_r.csv"), index=False)
    
mismatched_relationships

## Write Neo4j header files for bulk import

In [ ]:
def save_node_header(name, import_header, NEO4J_IMPORT):
    df = pd.DataFrame([], columns=import_header.split(","))
    df.to_csv(os.path.join(NEO4J_IMPORT, "header_" + name + "_n.csv"), index=False)

Write node header files

In [ ]:
matched_nodes.query("match == True", inplace=True)

In [ ]:
out = matched_nodes.apply(lambda row: save_node_header(row["node"], row["importHeader"], NEO4J_IMPORT), axis=1)

Write relationship header files

In [ ]:
def save_relationship_header(name, import_header, NEO4J_IMPORT):
    df = pd.DataFrame([], columns=import_header.split(","))
    df.to_csv(os.path.join(NEO4J_IMPORT, "header_" + name + "_r.csv"), index=False)

In [ ]:
matched_relationships.query("match == True", inplace=True)

In [ ]:
matched_relationships

In [ ]:
out = matched_relationships.apply(lambda row: save_relationship_header(row["fullRelationship"], row["importHeader"], NEO4J_IMPORT), axis=1)

## Create MetaNode and MetaRelationship files

Compile a dictionary of all node properties

In [ ]:
property_dir = {}
for header in matched_nodes["metadataHeader"]:
    for prop in header.split(","):
        property_dir[prop] = ""

Create dataframe with MetaNode data

In [ ]:
node_list = []
for node, filepath in matched_nodes[["node","metadataPath"]].itertuples(index=False):
    node_list.append(create_meta_node(node, property_dir, filepath))
    
meta_nodes = pd.DataFrame(node_list)
meta_nodes.drop_duplicates(inplace=True)
meta_nodes.to_csv(os.path.join(NEO4J_IMPORT, "MetaNode_n.csv"), index=False)

In [ ]:
meta_nodes

Compile a dictionary of all relationship properties

In [ ]:
property_dir = {}
for header in matched_relationships["metadataHeader"]:
    for property in header.split(","):
        property_dir[property] = ""

Create dataframe with MetaRelationship data

In [ ]:
relationship_list = []
for relationship, source, target, filepath in matched_relationships[["relationship", "source", "target", "metadataPath"]].itertuples(index=False):
    relationship_list.append(create_meta_relationship(relationship, source, target, property_dir, filepath))
    
meta_relationships = pd.DataFrame(relationship_list)
meta_relationships.drop_duplicates(inplace=True)
meta_relationships.to_csv(os.path.join(NEO4J_IMPORT, "MetaRelationship_r.csv"), index=False)

In [ ]:
meta_relationships

## Create Neo4j bulk upload command line arguments
See: https://neo4j.com/docs/operations-manual/current/import/

In [ ]:
args = ""
for node in matched_nodes["node"].unique():
    #args += f" --nodes={node}=header_{node}_n.csv,{node}*_n.csv"
    args += f" --nodes={node}=./header_{node}_n.csv,./{node}.*_n.csv"

In [ ]:
args += " --nodes=MetaNode=./MetaNode_n.csv"

In [ ]:
rel_data = matched_relationships[["relationship", "fullRelationship"]].copy()
rel_data.drop_duplicates(inplace=True)
for relationship, fullRelationship in rel_data.itertuples(index=False):
    #args += f" --relationships={relationship}=header_{fullRelationship}_r.csv,{fullRelationship}*_r.csv"
    args += f" --relationships={relationship}=./header_{fullRelationship}_r.csv,./{fullRelationship}.*_r.csv"

In [ ]:
args += " --relationships=MetaRelationship=./MetaRelationship_r.csv"

In [ ]:
file = open(os.path.join(NEO4J_IMPORT, "args.txt"), "w")
file.write(args)
file.close() 

In [ ]:
print(args)

## Create a Cypher script with default Constraints, indices, and fulltext indices

Create constraints for all node ids and indices and fulltext indices for all string properties

In [ ]:
indexed_nodes = matched_nodes[["node", "index", "stringProperties"]].copy()

In [ ]:
indices = "".join(indexed_nodes["index"].unique())

Create fulltext index

In [ ]:
node_names = indexed_nodes["node"].values
node_names = list(set(node_names))
node_names = "|".join(f"{w}" for w in node_names)

In [ ]:
property_names = indexed_nodes["stringProperties"].values
property_names = list(np.unique(np.concatenate(property_names).flat))
property_names = ",n.".join(f"{w}" for w in property_names)

In [ ]:
fulltext_index = f"CREATE FULLTEXT INDEX fulltext FOR (n:{node_names}) ON EACH [n.{property_names}];"

In [ ]:
indices = indices + fulltext_index
indices = indices.replace(";", ";\n")

In [ ]:
print(indices)

In [ ]:
with open(os.path.join(NEO4J_IMPORT, "indices.cypher"), "w") as f:
    f.write(indices)